# Generación de descripciones para imágenes con métodos XAI

Este notebook recorre la carpeta `imagenes/`, identifica las imágenes originales y las salidas XAI en subcarpetas, genera una descripción textual por imagen/método y guarda los resultados en `generacion_descripcion_XAI/resultados_descripciones_xai/`.

El flujo está pensado para las subcarpetas actuales del proyecto: `original`, `output_gradcam`, `output_lime`, `output_ig`, `output_saliency` y `output_anchor`.

## 0. Instalación opcional

Si falta alguna dependencia, ejecuta la celda siguiente. Para generar descripciones semánticas de las imágenes se usa Ollama con un modelo de visión local.

In [34]:
# Descomenta si necesitas instalar dependencias en el entorno del notebook.
# %pip install -q pandas tqdm openpyxl ollama

## 1. Imports y configuración

In [35]:
from pathlib import Path
import warnings

import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "Imagenes").exists() and (PROJECT_DIR.parent / "Imagenes").exists():
    PROJECT_DIR = PROJECT_DIR.parent

BASE_DIR = PROJECT_DIR / "Imagenes"
OUT_DIR = PROJECT_DIR / "generacion_descripcion_XAI" / "resultados_descripciones_xai"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

METHOD_FOLDERS = {
    "original": "original",
    "gradcam": "output_gradcam",
    "lime": "output_lime",
    "integrated_gradients": "output_ig",
    "saliency": "output_saliency",
    "anchor": "output_anchor",
}

assert BASE_DIR.exists(), f"No existe la carpeta esperada: {BASE_DIR.resolve()}"

## 2. Cargar rutas de imágenes y asociarlas por caso

La función de normalización usa los nombres de las imágenes originales como referencia.

In [36]:
def list_images(folder: Path):
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def strip_method_suffix(stem: str) -> str:
    suffixes = [
        "_gradcam",
        "_lime",
        "_ig",
        "_saliency",
        "_anchor",
        "_comparativa_vertical",
        "_comparativa",
    ]
    for suffix in suffixes:
        if stem.endswith(suffix):
            return stem[: -len(suffix)]
    return stem


original_paths = list_images(BASE_DIR / METHOD_FOLDERS["original"])
original_stems = sorted([p.stem for p in original_paths], key=len, reverse=True)


def infer_case_id(path: Path) -> str:
    clean_stem = strip_method_suffix(path.stem)
    for original_stem in original_stems:
        if original_stem == clean_stem or original_stem in clean_stem:
            return original_stem
    return clean_stem


records = []
for method, folder_name in METHOD_FOLDERS.items():
    folder = BASE_DIR / folder_name
    for path in list_images(folder):
        records.append(
            {
                "case_id": infer_case_id(path),
                "method": method,
                "folder": folder_name,
                "image_path": str(path),
                "file_name": path.name,
            }
        )

df_images = pd.DataFrame(records).sort_values(["case_id", "method"]).reset_index(drop=True)
display(df_images.head(20))
print(f"Imágenes encontradas: {len(df_images)}")
print(f"Casos detectados: {df_images['case_id'].nunique()}")
display(pd.crosstab(df_images["case_id"], df_images["method"]).head())

,case_id,method,folder,image_path,file_name
0,image01,anchor,output_anchor,/Users/haojie/PycharmProjects/TFM-Personalized...,image01_anchor.png
1,image01,gradcam,output_gradcam,/Users/haojie/PycharmProjects/TFM-Personalized...,image01_gradcam.png
2,image01,integrated_gradients,output_ig,/Users/haojie/PycharmProjects/TFM-Personalized...,image01_ig.png
3,image01,lime,output_lime,/Users/haojie/PycharmProjects/TFM-Personalized...,image01_lime.png
4,image01,original,original,/Users/haojie/PycharmProjects/TFM-Personalized...,image01.jpg
5,image01,saliency,output_saliency,/Users/haojie/PycharmProjects/TFM-Personalized...,image01_saliency.png
6,image02,anchor,output_anchor,/Users/haojie/PycharmProjects/TFM-Personalized...,image02_anchor.png
7,image02,gradcam,output_gradcam,/Users/haojie/PycharmProjects/TFM-Personalized...,image02_gradcam.png
8,image02,integrated_gradients,output_ig,/Users/haojie/PycharmProjects/TFM-Personalized...,image02_ig.png
9,image02,lime,output_lime,/Users/haojie/PycharmProjects/TFM-Personalized...,image02_lime.png


Imágenes encontradas: 66
Casos detectados: 11


method,anchor,gradcam,integrated_gradients,lime,original,saliency
case_id,,,,,,
image01,1,1,1,1,1,1
image02,1,1,1,1,1,1
image03,1,1,1,1,1,1
image04,1,1,1,1,1,1
image05,1,1,1,1,1,1


## 3. Configurar modelo local para describir imágenes

Ollama con `qwen2.5vl:32b`

Antes de ejecutar esta sección, comprueba que Ollama esté arrancado y que el modelo exista con `ollama show qwen2.5vl:32b`. Las descripciones se generan en inglés para mantener una salida homogénea en los textos usados posteriormente.

In [37]:
USE_OLLAMA = True
OLLAMA_MODEL = "qwen2.5vl:32b"

# Importante: este modelo declara un contexto máximo enorme (262144).
# Para describir imágenes cortas no hace falta y puede hacer que la primera llamada tarde muchísimo.
OLLAMA_OPTIONS = {
    "temperature": 0.0,
    "top_p": 0.85,
    "num_ctx": 2048,
    "num_predict": 320,
}

OLLAMA_PYTHON_AVAILABLE = False
if USE_OLLAMA:
    try:
        import ollama  # noqa: F401
        OLLAMA_PYTHON_AVAILABLE = True
    except Exception as exc:
        print(
            "No se encontró el paquete Python 'ollama'. "
            "Instalalo con: pip install ollama (o uv add ollama)."
        )
        print(f"Detalle: {type(exc).__name__}: {str(exc)[:200]}")
        USE_OLLAMA = False

if not USE_OLLAMA:
    raise RuntimeError(
        "No hay backend para captions: Ollama está desactivado o no está disponible. "
        "Instala/configura Ollama y el paquete Python 'ollama'."
    )

REQUIRE_ENGLISH_CAPTIONS = True


def looks_spanish_text(text: str) -> bool:
    lowered = f" {(text or '').lower()} "
    spanish_markers = [
        " la imagen ", " el modelo ", " una visualizacion ", " una visualización ",
        " en espanol", " en español", " zonas resaltadas", " clase predicha",
        " deteccion ", " detección ", " aprendizaje ", " tortuga ", " rinc ",
    ]
    return any(marker in lowered for marker in spanish_markers)


def looks_degenerate_text(text: str) -> bool:
    t = (text or "").replace("\u00ad", "").strip()
    if not t:
        return True
    if len(t) < 25:
        return True
    lowered = t.lower()
    if REQUIRE_ENGLISH_CAPTIONS and looks_spanish_text(t):
        return True
    bad_fragments = [
        "[img-0]", "de la com", "de neu", "deep de neu", "1.1.1.1.1", "[  ]",
        "­­­­", " la tort", "\" \" \"", "“ “ “", "… …",
    ]
    if any(f in lowered for f in bad_fragments):
        return True
    compact = " ".join(lowered.split())
    if compact:
        # Detecta bucles de una palabra y repeticiones de n-gramas cortos.
        words = compact.split()
        for n, threshold in ((1, 6), (2, 5), (3, 4), (4, 4), (5, 4)):
            if len(words) >= n * threshold:
                chunks = [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]
                top = max(chunks.count(ch) for ch in set(chunks))
                if top >= threshold:
                    return True
    quote_count = sum(t.count(ch) for ch in ['"', "'", "“", "”"])
    if quote_count >= 8:
        return True
    return False


def looks_incomplete_text(text: str) -> bool:
    t = clean_caption_text(text)
    if not t:
        return True
    lowered = t.lower().strip()
    incomplete_fragments = [
        "[incomplete response]", "[...]", "…", " etc", " such as the ,",
        " the and ", " and .", " or .", " is .", " are .", " most .", " more or .",
        "to represent the", "with a super-", "with a split-", "corresponding sali",
        "with sali", "is a sali", "areas of sali", "road-",
    ]
    if any(fragment in lowered for fragment in incomplete_fragments):
        return True
    if lowered.endswith(("-", ",", ":", ";", " and", " or", " the", " a", " an", " of", " with", " to", " for", " in", " by", " sali", " super", " split")):
        return True
    if t[-1] not in ".!?":
        return True
    words = lowered.rstrip(".!?").split()
    if words and words[-1] in {"the", "a", "an", "of", "with", "to", "for", "in", "by", "and", "or", "sali", "super", "split"}:
        return True
    return False


def is_valid_caption(text: str) -> bool:
    return not looks_degenerate_text(text) and not looks_incomplete_text(text)


def clean_caption_text(text: str) -> str:
    t = (text or "").replace("\u00ad", "").replace("\n", " ").strip()
    t = " ".join(t.split())
    if len(t) > 700:
        t = t[:700].rsplit(" ", 1)[0].rstrip(",;:-") + "."
    return t


def build_prompt_for_method(method: str) -> str:
    if method == "original":
        return (
            "Describe the original image in English in one complete sentence of 20 to 45 words. "
            "Mention only visible objects or scene elements. Do not invent information. "
            "Do not use lists or markdown. End with a complete sentence."
        )
    if method == "comparativa":
        return (
            "This is a composite comparison panel with an original image and several XAI views. "
            "Describe in English, in one complete sentence of 25 to 60 words, which panels or highlighted regions stand out and whether the explanations look consistent. "
            "Do not list every panel. Do not use markdown. End with a complete sentence."
        )
    return (
        "Describe this XAI visualization in English in one complete sentence of 25 to 55 words. "
        "Mention the visible highlighted areas (warm, bright, masked, or marked regions) and the general interpretation they suggest. "
        "Do not identify uncertain brands or unclear text. Do not use lists or markdown. End with a complete sentence."
    )


def ollama_caption_attempt(path: str, prompt: str, attempt_instruction: str, num_predict: int = 420) -> str:
    import ollama

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You write concise, complete image captions only in English. Never answer in Spanish. Never repeat words or phrases. Always finish the final sentence.",
            },
            {
                "role": "user",
                "content": prompt + " " + attempt_instruction,
                "images": [str(Path(path).resolve())],
            },
        ],
        think=False,
        options={**OLLAMA_OPTIONS, "num_predict": num_predict, "temperature": 0.0, "top_p": 0.8},
        keep_alive="5m",
    )
    return clean_caption_text(response["message"]["content"])


def caption_image(path: str, method: str = "original") -> str:
    prompt = build_prompt_for_method(method)
    attempt_instructions = [
        "Return only the final caption.",
        "Try again with a shorter complete caption of 20 to 40 words. Return only the caption.",
        "Describe only the visible colors, masks, heatmaps, and highlighted regions. Return one complete sentence.",
        "If the image is complex, give a generic but image-specific XAI caption based only on visible regions. Return one complete sentence.",
    ]
    try:
        for attempt_idx, instruction in enumerate(attempt_instructions, start=1):
            caption = ollama_caption_attempt(path, prompt, instruction, num_predict=420)
            if is_valid_caption(caption):
                return caption
            print(f"Caption rejected attempt {attempt_idx}/{len(attempt_instructions)} for {Path(path).name}: {caption[:180]}", flush=True)
    except Exception as exc:
        print(f"Ollama fallo con {path}: {type(exc).__name__}: {str(exc)[:200]}")
    return ""


### Comprobación de Ollama y GPU/Metal

En Mac Apple Silicon, Ollama usa Metal/GPU automáticamente cuando el modelo y la memoria lo permiten. Esta celda comprueba que el modelo existe y recuerda cómo verificar el uso real de GPU durante una inferencia.

In [38]:
import subprocess


def run_ollama_command(args):
    try:
        result = subprocess.run(args, capture_output=True, text=True, check=False)
        return result.returncode, result.stdout.strip(), result.stderr.strip()
    except FileNotFoundError:
        return 127, "", "No se encontró el comando 'ollama'. Instala Ollama o abre Jupyter desde una terminal donde esté disponible."


code, stdout, stderr = run_ollama_command(["ollama", "show", OLLAMA_MODEL])
if code == 0:
    print(f"Modelo Ollama encontrado: {OLLAMA_MODEL}")
    capability_lines = [line for line in stdout.splitlines() if "vision" in line.lower() or "requires" in line.lower()]
    if capability_lines:
        print("\n".join(capability_lines))
else:
    print(f"No se pudo comprobar el modelo {OLLAMA_MODEL}.")
    print(stderr or stdout)

print("\nDurante la ejecución de captions, abre otra terminal y usa: ollama ps")
print("Opciones Ollama usadas:", OLLAMA_OPTIONS)

Modelo Ollama encontrado: qwen2.5vl:32b
    vision        

Durante la ejecución de captions, abre otra terminal y usa: ollama ps
Opciones Ollama usadas: {'temperature': 0.0, 'top_p': 0.85, 'num_ctx': 2048, 'num_predict': 320}


### Prueba rápida con una sola imagen

Activa esta celda antes del lote completo si quieres validar una imagen aislada. Si tarda varios minutos, el problema está en el modelo/configuración y no en el bucle.

In [39]:
import time

TEST_SINGLE_IMAGE = False

if TEST_SINGLE_IMAGE:
    test_row = df_images[df_images["method"] == "original"].iloc[0]
    print("Imagen de prueba:", test_row["image_path"])
    start = time.perf_counter()
    test_caption = caption_image(test_row["image_path"], method="original")
    elapsed = time.perf_counter() - start
    print(f"Tiempo: {elapsed:.1f}s")
    print(test_caption)

## 4. Leer metadatos de Anchor y utilidades de descripción

In [40]:
def parse_anchor_metadata(case_id: str):
    folder = BASE_DIR / METHOD_FOLDERS["anchor"]
    if not folder.exists():
        return {}
    candidates = [p for p in folder.glob("*.txt") if case_id in p.stem]
    if not candidates:
        return {}

    metadata = {"anchor_txt_path": str(candidates[0])}
    text = candidates[0].read_text(encoding="utf-8", errors="ignore")
    for line in text.splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        metadata[key.strip()] = value.strip()
    return metadata


def format_prob(value):
    try:
        return f"{float(value):.3f}"
    except Exception:
        return value


METHOD_EXPLANATIONS = {
    "original": "Original image used as the reference for comparing the XAI explanations.",
    "gradcam": "Grad-CAM highlights the warm regions of the map as the most relevant areas, indicating the parts of the image that most influenced the predicted class.",
    "lime": "LIME divides the image into superpixels and marks the regions that most support the model prediction.",
    "integrated_gradients": "Integrated Gradients assigns importance to pixels by comparing the image with a baseline reference; the most intense regions concentrate the strongest contribution to the prediction.",
    "saliency": "Saliency Maps highlight the pixels where small changes would most affect the model output; brighter areas indicate greater sensitivity.",
    "anchor": "Anchor Image identifies a set of superpixels that act as a sufficient rule for preserving the model prediction.",
    "comparativa": "Comparative image that groups the original image and several XAI explanations to review the consistency of the methods together.",
}


METHOD_FALLBACK_CAPTIONS = {
    "gradcam": "The Grad-CAM output should be interpreted as a heatmap where warmer regions indicate the image areas that contributed most strongly to the predicted class.",
    "lime": "The LIME output should be interpreted as a superpixel-based explanation where the marked regions indicate the image segments that most support the model prediction.",
    "integrated_gradients": "The Integrated Gradients output should be interpreted as an attribution map where the most intense regions indicate pixels with the strongest contribution relative to a baseline image.",
    "saliency": "The Saliency Map output should be interpreted as a sensitivity map where brighter regions indicate pixels whose small changes would most affect the model output.",
    "anchor": "The Anchor Image output should be interpreted as a masked superpixel explanation where the visible selected regions form a sufficient visual rule for preserving the prediction.",
    "comparativa": "The comparative XAI output should be interpreted as a side-by-side summary that allows the original image and multiple explanation methods to be checked for consistency.",
}


def build_fallback_xai_caption(row, original_caption_by_case, anchor_meta_by_case):
    method = row["method"]
    if method == "original":
        return ""

    fallback = METHOD_FALLBACK_CAPTIONS.get(method, "This XAI output should be interpreted as a visual explanation of the regions used by the model for its prediction.")
    base_caption = original_caption_by_case.get(row["case_id"], "")
    if base_caption:
        fallback += f" The explanation belongs to a base image described as: {base_caption}."

    if method == "anchor":
        meta = anchor_meta_by_case.get(row["case_id"], {})
        anchor_bits = []
        if meta.get("pred_class_name"):
            anchor_bits.append(f"predicted class '{meta['pred_class_name']}'")
        if meta.get("precision"):
            anchor_bits.append(f"precision {format_prob(meta['precision'])}")
        if meta.get("coverage"):
            anchor_bits.append(f"coverage {format_prob(meta['coverage'])}")
        if meta.get("anchor_features (superpixels)"):
            anchor_bits.append(f"anchor superpixels {meta['anchor_features (superpixels)']}")
        if anchor_bits:
            fallback += " Anchor metadata: " + ", ".join(anchor_bits) + "."

    return clean_caption_text(fallback)


def build_xai_description(row, original_caption_by_case, xai_caption_by_path, anchor_meta_by_case):
    method = row["method"]
    case_id = row["case_id"]
    base_caption = original_caption_by_case.get(case_id, "")
    xai_caption = xai_caption_by_path.get(row["image_path"], "")
    if method != "original" and not str(xai_caption).strip():
        xai_caption = build_fallback_xai_caption(row, original_caption_by_case, anchor_meta_by_case)
    method_text = METHOD_EXPLANATIONS.get(method, "XAI output associated with the image.")

    pieces = []
    if base_caption:
        pieces.append(f"The base image shows: {base_caption}.")
    if method == "original":
        pieces.append(method_text)
    else:
        pieces.append(f"Method {method}: {method_text}")
    if method != "original" and xai_caption and xai_caption != base_caption:
        pieces.append(f"The generated visualization can be visually described as: {xai_caption}.")

    if method == "anchor":
        meta = anchor_meta_by_case.get(case_id, {})
        pred_name = meta.get("pred_class_name")
        pred_prob = meta.get("pred_prob")
        precision = meta.get("precision")
        coverage = meta.get("coverage")
        features = meta.get("anchor_features (superpixels)")
        anchor_details = []
        if pred_name:
            anchor_details.append(f"predicted class '{pred_name}'")
        if pred_prob:
            anchor_details.append(f"probability {format_prob(pred_prob)}")
        if precision:
            anchor_details.append(f"precision {format_prob(precision)}")
        if coverage:
            anchor_details.append(f"coverage {format_prob(coverage)}")
        if features:
            anchor_details.append(f"anchor superpixels {features}")
        if anchor_details:
            pieces.append("Anchor metadata: " + ", ".join(anchor_details) + ".")

    return " ".join(pieces).strip()


anchor_meta_by_case = {case_id: parse_anchor_metadata(case_id) for case_id in df_images["case_id"].unique()}

## 5. Generar captions y descripciones

Esta sección se detiene si detecta que varias imágenes originales reciben exactamente la misma descripción, porque eso suele indicar que el modelo de visión no está leyendo las imágenes.

In [41]:
import time

SHOW_IMAGE_PROGRESS = True
CAPTION_PROGRESS_PREVIEW_CHARS = 700
APPLY_FALLBACK_CAPTIONS = False


def caption_rows_with_progress(rows: pd.DataFrame, desc: str):
    captions = {}
    total = len(rows)
    iterator = tqdm(rows.reset_index(drop=True).iterrows(), total=total, desc=desc)
    for idx, (_, row) in enumerate(iterator, start=1):
        image_path = row["image_path"]
        method = row["method"]
        case_id = row["case_id"]
        iterator.set_postfix_str(f"{idx}/{total} {method} {Path(image_path).name[:35]}")

        if SHOW_IMAGE_PROGRESS:
            print(f"[{idx}/{total}] Procesando {method} | caso={case_id} | archivo={Path(image_path).name}", flush=True)

        start = time.perf_counter()
        caption = caption_image(image_path, method=method)
        elapsed = time.perf_counter() - start

        if SHOW_IMAGE_PROGRESS:
            preview = caption.replace("\n", " ")[:CAPTION_PROGRESS_PREVIEW_CHARS]
            status = "OK" if caption else "REJECTED/EMPTY"
            print(f"[{idx}/{total}] Terminado en {elapsed:.1f}s | {status} | {preview}", flush=True)

        caption = clean_caption_text(caption)
        captions[image_path] = caption
    return captions


original_rows = df_images[df_images["method"] == "original"]
original_caption_by_path = caption_rows_with_progress(original_rows, "Captions originales")
original_caption_by_case = dict(zip(original_rows["case_id"], original_rows["image_path"].map(original_caption_by_path)))

unique_original_captions = {caption for caption in original_caption_by_case.values() if str(caption).strip()}
if len(original_caption_by_case) > 1 and len(unique_original_captions) <= 1:
    raise RuntimeError(
        "Todas las imágenes originales han recibido el mismo caption. "
        "Esto indica que el modelo de visión no está interpretando las imágenes correctamente; "
        "detén la ejecución, revisa Ollama/modelo y regenera los resultados."
    )


# Con Ollama vision conviene describir tambien cada salida XAI.
# Si va demasiado lento con el modelo de visión local, cambia a False.
CAPTION_XAI_OUTPUTS = True
xai_caption_by_path = {}

if CAPTION_XAI_OUTPUTS:
    xai_rows = df_images[df_images["method"] != "original"]
    xai_caption_by_path = caption_rows_with_progress(xai_rows, "Captions XAI")

xai_caption_series = pd.Series(xai_caption_by_path, dtype="object")
xai_caption_ok = xai_caption_series.fillna("").astype(str).str.strip().ne("")
coverage_ratio = xai_caption_ok.mean() if len(xai_caption_ok) else 1.0

if CAPTION_XAI_OUTPUTS and coverage_ratio < 0.8:
    missing_xai_paths = xai_caption_series.index[~xai_caption_ok].tolist()
    missing_xai_rows = df_images[df_images["image_path"].isin(missing_xai_paths)]
    print(
        f"Warning: low XAI caption coverage ({coverage_ratio:.1%}). "
        "The notebook will continue using the method explanation as a fallback for those rows."
    )
    if not missing_xai_rows.empty:
        display(missing_xai_rows[["case_id", "method", "file_name"]].sort_values(["method", "case_id"]))

if APPLY_FALLBACK_CAPTIONS:
    for _, row in df_images[df_images["method"] != "original"].iterrows():
        image_path = row["image_path"]
        if not str(xai_caption_by_path.get(image_path, "")).strip():
            xai_caption_by_path[image_path] = build_fallback_xai_caption(row, original_caption_by_case, anchor_meta_by_case)
    print(f"Fallback XAI captions applied where needed. Non-empty XAI captions: {sum(bool(str(v).strip()) for v in xai_caption_by_path.values())}/{len(xai_caption_by_path)}")
else:
    print(f"Fallback disabled. Model-generated non-empty XAI captions: {sum(bool(str(v).strip()) for v in xai_caption_by_path.values())}/{len(xai_caption_by_path)}")
    pending_model_caption_paths = [path for path, caption in xai_caption_by_path.items() if not str(caption).strip()]
    pending_model_caption_rows = df_images[df_images["image_path"].isin(pending_model_caption_paths)]
    if not pending_model_caption_rows.empty:
        display(pending_model_caption_rows[["case_id", "method", "file_name", "image_path"]].sort_values(["method", "case_id"]))

df_descriptions = df_images.copy()
df_descriptions["caption_original"] = df_descriptions["case_id"].map(original_caption_by_case)
df_descriptions["caption_xai"] = df_descriptions["image_path"].map(xai_caption_by_path).fillna("")
df_descriptions["descripcion"] = df_descriptions.apply(
    lambda row: build_xai_description(row, original_caption_by_case, xai_caption_by_path, anchor_meta_by_case),
    axis=1,
)

display(df_descriptions[["case_id", "method", "image_path", "descripcion"]].head(20))

Captions originales:   0%|          | 0/11 [00:00<?, ?it/s]

[1/11] Procesando original | caso=image01 | archivo=image01.jpg


[1/11] Terminado en 20.5s | OK | A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward.
[2/11] Procesando original | caso=image02 | archivo=image02.jpg
[2/11] Terminado en 14.1s | OK | A close-up profile of a duck showcases its vibrant green head, contrasting black and brown feathers, and a bright yellow beak, set against a plain white background.
[3/11] Procesando original | caso=image03 | archivo=image03.jpg
[3/11] Terminado en 13.7s | OK | A snake with a dark, glossy body and a lighter underside coils gracefully on a rough, light-colored gravel surface, casting a subtle shadow.
[4/11] Procesando original | caso=image04 | archivo=image04.jpg
[4/11] Terminado en 19.1s | OK | A small spider with a dark, glossy body and slender legs is perched on a green leaf with white speckles, set against a neutral background with a scale bar in the top left corner.
[5/11] Procesando ori

Captions XAI:   0%|          | 0/55 [00:00<?, ?it/s]

[1/55] Procesando anchor | caso=image01 | archivo=image01_anchor.png
[1/55] Terminado en 12.4s | OK | This XAI visualization compares an original image of a bird in flight with its masked anchor representation, segmented superpixels, and an overlay indicating a kite detection with high precision but zero coverage, highlighting the model's focus on specific regions while missing the broader context.
[2/55] Procesando gradcam | caso=image01 | archivo=image01_gradcam.png
[2/55] Terminado en 11.7s | OK | This XAI visualization compares an original image of a bird in flight with a heatmap and an overlay, highlighting a warm, bright region around the bird's body and wings, suggesting that the model's attention is focused on these areas, likely indicating the bird's identification as a kite with a logit score of 6.43.
[3/55] Procesando integrated_gradients | caso=image01 | archivo=image01_ig.png
[3/55] Terminado en 13.9s | OK | This XAI visualization compares an original image of a bird in fl

,case_id,method,image_path,descripcion
0,image01,anchor,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
1,image01,gradcam,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
2,image01,integrated_gradients,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
3,image01,lime,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
4,image01,original,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
5,image01,saliency,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
6,image02,anchor,/Users/haojie/PycharmProjects/TFM-Personalized...,The base image shows: A close-up profile of a ...
7,image02,gradcam,/Users/haojie/PycharmProjects/TFM-Personalized...,The base image shows: A close-up profile of a ...
8,image02,integrated_gradients,/Users/haojie/PycharmProjects/TFM-Personalized...,The base image shows: A close-up profile of a ...
9,image02,lime,/Users/haojie/PycharmProjects/TFM-Personalized...,The base image shows: A close-up profile of a ...


## 6. Crear resumen por caso

Además de una fila por imagen/método, se crea un texto compacto por caso con todas las explicaciones disponibles.

In [42]:
def build_case_summary(group: pd.DataFrame) -> str:
    case_id = group.name
    caption = group["caption_original"].dropna().astype(str).replace("", pd.NA).dropna()
    header = f"Case {case_id}."
    if len(caption):
        header += f" Original image: {caption.iloc[0]}."

    method_lines = []
    for method in ["gradcam", "lime", "integrated_gradients", "saliency", "anchor"]:
        rows = group[group["method"] == method]
        if rows.empty:
            continue
        desc = rows["descripcion"].iloc[0]
        method_lines.append(f"- {method}: {desc}")
    return header + "\n" + "\n".join(method_lines)


df_case_summaries = (
    df_descriptions.groupby("case_id", sort=True)
    .apply(build_case_summary, include_groups=False)
    .reset_index(name="resumen_xai")
)

display(df_case_summaries.head())

,case_id,resumen_xai
0,image01,"Case image01. Original image: A bird of prey, ..."
1,image02,Case image02. Original image: A close-up profi...
2,image03,Case image03. Original image: A snake with a d...
3,image04,Case image04. Original image: A small spider w...
4,image05,Case image05. Original image: A vibrant blue b...


## 7. Guardar resultados

In [43]:
csv_path = OUT_DIR / "descripciones_por_imagen_xai.csv"
xlsx_path = OUT_DIR / "descripciones_por_imagen_xai.xlsx"
summary_csv_path = OUT_DIR / "resumenes_por_caso_xai.csv"


df_descriptions.to_csv(csv_path, index=False, encoding="utf-8")
df_case_summaries.to_csv(summary_csv_path, index=False, encoding="utf-8")

try:
    df_descriptions.to_excel(xlsx_path, index=False)
except Exception as exc:
    print("No se pudo guardar XLSX. Instala openpyxl si lo necesitas: %pip install openpyxl")
    print(type(exc).__name__, str(exc)[:200])



print("Archivos generados:")
print(csv_path)
print(summary_csv_path)

if xlsx_path.exists():
    print(xlsx_path)

Archivos generados:
/Users/haojie/PycharmProjects/TFM-Personalized-XAI/generacion_descripcion_XAI/resultados_descripciones_xai/descripciones_por_imagen_xai.csv
/Users/haojie/PycharmProjects/TFM-Personalized-XAI/generacion_descripcion_XAI/resultados_descripciones_xai/resumenes_por_caso_xai.csv
/Users/haojie/PycharmProjects/TFM-Personalized-XAI/generacion_descripcion_XAI/resultados_descripciones_xai/descripciones_por_imagen_xai.xlsx


## 8. Vista rápida de un caso

In [44]:
CASE_ID = df_case_summaries["case_id"].iloc[0]

display(Markdown(df_case_summaries.loc[df_case_summaries["case_id"] == CASE_ID, "resumen_xai"].iloc[0]))
display(df_descriptions[df_descriptions["case_id"] == CASE_ID][["method", "image_path", "descripcion"]])

Case image01. Original image: A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward..
- gradcam: The base image shows: A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward.. Method gradcam: Grad-CAM highlights the warm regions of the map as the most relevant areas, indicating the parts of the image that most influenced the predicted class. The generated visualization can be visually described as: This XAI visualization compares an original image of a bird in flight with a heatmap and an overlay, highlighting a warm, bright region around the bird's body and wings, suggesting that the model's attention is focused on these areas, likely indicating the bird's identification as a kite with a logit score of 6.43..
- lime: The base image shows: A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward.. Method lime: LIME divides the image into superpixels and marks the regions that most support the model prediction. The generated visualization can be visually described as: This XAI visualization compares an original image of a bird in flight with a superpixel mask and a LIME kite explanation, highlighting the bird's silhouette and key features in warm, bright, and marked regions, suggesting the model's focus on the bird's shape and edges for accurate detection..
- integrated_gradients: The base image shows: A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward.. Method integrated_gradients: Integrated Gradients assigns importance to pixels by comparing the image with a baseline reference; the most intense regions concentrate the strongest contribution to the prediction. The generated visualization can be visually described as: This XAI visualization compares an original image of a bird in flight with its corresponding IG |abs| (gris) grayscale representation, IG heatmap, and an overlay with a kite (0.28) score, highlighting the bird's wings and body as the most influential features for model predictions, suggesting these areas are critical for accurate classification..
- saliency: The base image shows: A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward.. Method saliency: Saliency Maps highlight the pixels where small changes would most affect the model output; brighter areas indicate greater sensitivity. The generated visualization can be visually described as: This XAI visualization compares an original image of a bird in flight with a saliency map highlighting areas of high importance, and an overlay indicating the presence of a kite with a logit score of 6.43, suggesting the model's focus on the bird's wings and body as key features for identifying the kite..
- anchor: The base image shows: A bird of prey, with dark brown and white plumage, soars gracefully through a clear blue sky, its wings fully extended and tail feathers fanned out as it glides forward.. Method anchor: Anchor Image identifies a set of superpixels that act as a sufficient rule for preserving the model prediction. The generated visualization can be visually described as: This XAI visualization compares an original image of a bird in flight with its masked anchor representation, segmented superpixels, and an overlay indicating a kite detection with high precision but zero coverage, highlighting the model's focus on specific regions while missing the broader context.. Anchor metadata: predicted class 'kite', probability 0.276, precision 1.000, coverage 0.001, anchor superpixels [2, 6, 1, 4, 12, 9, 11, 13, 7, 10].

,method,image_path,descripcion
0,anchor,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
1,gradcam,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
2,integrated_gradients,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
3,lime,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
4,original,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
5,saliency,/Users/haojie/PycharmProjects/TFM-Personalized...,"The base image shows: A bird of prey, with dar..."
